In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, paths
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "ethiopia"
vehicle = "salt"
has_simulations = False
has_anemia_model = True
has_ntd_model = True


In [4]:
rescaled_pregnancy_root = paths.source_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT, data_source)
rescaled_child_root = paths.source_root(paths.RESCALED_CHILD_RESULTS_ROOT, data_source)
anemia_root = paths.source_root(paths.NON_PREGNANT_ANEMIA_RESULTS_ROOT, data_source)
print(f"{data_source}:\n  {rescaled_pregnancy_root}\n  {rescaled_child_root}\n  {anemia_root}")

model2.6.3:
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_pregnancy
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_child
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/non_pregnant_anemia


In [5]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention_25_nrv',
 'intervention_100_nrv',
 'intervention_45_ppm',
 'zero',
 'baseline']

In [6]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylls.parquet"
if has_simulations:
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention_25_nrv,0,141,0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention_25_nrv,0,141,0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,intervention_25_nrv,0,141,0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,intervention_25_nrv,0,141,0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,intervention_25_nrv,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
538195,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,54,0
538196,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,54,0
538197,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,54,0
538198,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,54,0


In [7]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline                200
intervention_100_nrv    200
intervention_25_nrv     200
intervention_45_ppm     200
zero                    200
Name: random_seed, dtype: int64

In [8]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [9]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario              entity              wealth_quintile
baseline              maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_25_nrv   maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [10]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention_25_nrv,0,141,0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention_25_nrv,0,141,0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention_25_nrv,0,141,0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention_25_nrv,0,141,0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,intervention_25_nrv,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
1883695,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,54,0
1883696,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,54,0
1883697,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,54,0
1883698,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,54,0


In [11]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [12]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [13]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [14]:
ylds_path = f"{rescaled_child_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [15]:
path = f"{rescaled_child_root}/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # Combos with no iron pathway (e.g. ethiopia/salt, nigeria/salt) have no
    # Vivarium sims, so the corresponding DALYs are 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"{rescaled_child_root}/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention_25_nrv,0,81,0
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention_25_nrv,0,81,0
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention_25_nrv,0,81,0
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention_25_nrv,0,81,0
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention_25_nrv,0,81,0
...,...,...,...,...,...,...,...,...,...,...,...,...
23955,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,0,129,0
23956,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,0,129,0
23957,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,0,129,0
23958,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,0,129,0


In [16]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario              entity  wealth_quintile
baseline              lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_100_nrv  lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_25_nrv   lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_45_ppm   lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0


In [17]:
path = f"{anemia_root}/{vehicle}/{location}/ylds.parquet"
if has_anemia_model:
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{anemia_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,395.450077,zero
1,Female,0.0,0.019178,2,295.482645,zero
2,Female,0.0,0.019178,3,240.051599,zero
3,Female,0.0,0.019178,4,230.732019,zero
4,Female,0.0,0.019178,5,150.407217,zero
...,...,...,...,...,...,...
1245,Male,95.0,125.000000,4,26.059157,intervention_25_nrv
1246,Male,95.0,125.000000,4,26.059157,intervention_45_ppm
1247,Male,95.0,125.000000,5,18.786956,intervention_100_nrv
1248,Male,95.0,125.000000,5,18.786956,intervention_25_nrv


In [18]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  95864.262160
                              2                  64001.802423
                              3                  58573.327149
                              4                  54194.911297
                              5                  50608.699564
intervention_100_nrv  anemia  1                  78460.421963
                              2                  48613.944854
                              3                  41858.578025
                              4                  46871.471792
                              5                  44543.193175
intervention_25_nrv   anemia  1                  90980.266171
                              2                  59464.110528
                              3                  53471.095496
                              4                  52210.515470
                              5                  48992.784373
intervention_45_ppm   an

In [19]:
scenarios[1]

'intervention_100_nrv'

In [20]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

np.float64(62895.392783088115)

In [21]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  142971.456860
                              2                  100076.303330
                              3                   75209.047505
                              4                   73130.287443
                              5                   46295.820229
intervention_100_nrv  anemia  1                  142971.456860
                              2                  100076.303330
                              3                   75209.047505
                              4                   73130.287443
                              5                   46295.820229
intervention_25_nrv   anemia  1                  142971.456860
                              2                  100076.303330
                              3                   75209.047505
                              4                   73130.287443
                              5                   46295.820229
intervent

In [22]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

np.float64(0.0)

In [23]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  405464.960442
                              2                  263500.263289
                              3                  216719.525793
                              4                  209626.540341
                              5                  148517.132894
intervention_100_nrv  anemia  1                  379050.865088
                              2                  241189.778455
                              3                  192829.907919
                              4                  199208.164565
                              5                  140394.674113
intervention_25_nrv   anemia  1                  398032.871741
                              2                  256916.812515
                              3                  209433.050648
                              4                  206803.942946
                              5                  146354.901423
intervent

In [24]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if has_ntd_model:
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario              entity  wealth_quintile
zero                  ntd     1                  134233.084212
                              2                  127686.366683
                              3                  116216.923571
                              4                  102412.992038
                              5                   80793.599269
baseline              ntd     1                  134233.084212
                              2                  127686.366683
                              3                  116216.923571
                              4                  102412.992038
                              5                   80793.599269
intervention_25_nrv   ntd     1                   89352.617839
                              2                   82059.017375
                              3                   70492.383076
                              4                   81929.198546
                              5                   68950.982601
intervent

In [25]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario  entity  wealth_quintile
baseline  anemia  1                  405464.960442
                  2                  263500.263289
                  3                  216719.525793
                  4                  209626.540341
                  5                  148517.132894
                                         ...      
zero      ntd     1                  134233.084212
                  2                  127686.366683
                  3                  116216.923571
                  4                  102412.992038
                  5                   80793.599269
Name: value, Length: 100, dtype: float64

In [26]:
import pathlib

In [27]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)